# 00-07 - Reading Errors

Everybody who writes code produces errors, all day, permanently. Experienced programmers do not make fewer of them; they simply read them faster and stop being frightened by red text.

In this notebook we produce five errors on purpose. These are not a representative sample of everything Python can complain about, they are the five you will actually meet in this course, and after this notebook you will recognise all of them on sight. We finish with the thing nobody teaches and everybody needs, which is what to send somebody when you are stuck.

Every red block below is meant to be there. If you run this notebook and see red, it is working.

In [1]:
import pandas as pd

## How to read a traceback

When Python cannot do what you asked, it stops and prints a **traceback**, which is a report of where it got to before it gave up. Tracebacks look intimidating because they are long and because the important part is at the end rather than the beginning.

So the rule is simple: **read the last line first.**

The last line has two parts, separated by a colon. Before the colon is the **kind** of problem, which is one of a fairly small set of names. After the colon is the detail, i.e. which name, which column, which file.

Everything above the last line is the trail of where Python was when it stopped. In a notebook, most of that trail is inside libraries you did not write and cannot fix, so it is usually noise. The useful part is the arrow, `---->`, which marks the line in *your* cell that caused it.

Let us make one and read it together:

In [2]:
places_to_eat

NameError: name 'places_to_eat' is not defined

Read the last line. It says `NameError: name 'places_to_eat' is not defined`.

**`NameError` means Python does not recognise a word you used.** There are only two reasons, and they need different fixes.

The first is a typo, and the fix is a keystroke. The second is more common and more confusing: the name is spelled perfectly, but the cell that creates it has not been run in this session. You opened the notebook and started clicking in the middle, or you restarted the kernel and forgot, and the kernel genuinely has never heard of it.

That second case is why the standard first move, when a notebook misbehaves, is Restart Kernel and Run All Cells. Notebook 00-01 in this folder is entirely about why.

Let us fix ours by creating the name:

In [3]:
places_to_eat = 92
places_to_eat

92

## TypeError, i.e. the wrong kind of thing

Python cares about what kind of thing a value is. The number `92` and the text `"92"` look similar on the page and behave completely differently, because one is a quantity and the other is two characters.

Adding a number to a piece of text is not something Python is willing to guess about:

In [4]:
"92" + 1

TypeError: can only concatenate str (not "int") to str

`TypeError: can only concatenate str (not "int") to str`.

That message is written from Python's point of view, which takes a moment to get used to. `str` means text, i.e. a string. `int` means a whole number, i.e. an integer. And "concatenate" means gluing text together, which is what `+` does when both sides are text.

So it is saying: you asked me to glue something onto a piece of text, and the something was a number, and I will not assume what you meant.

**`TypeError` means you gave an operation the wrong kind of thing.** The fix is to convert one side so that both agree. `int()` turns text into a number:

In [5]:
print(int("92") + 1)
print("92" + "1")

93
921


Note that both of those ran, and they gave different answers: 93 and `921`. Neither is an error, and only one of them is probably what you wanted. That is worth remembering, because Python will not always stop you.

## KeyError, i.e. that column is not there

Now we load a real table, the places to eat in Princeton with the geometry stripped off, so it is an ordinary spreadsheet of names and coordinates:

In [6]:
food = pd.read_csv("data/food_table.csv")
food.head(3)

,name,amenity,lat,lon
0,Metro North,restaurant,40.33471,-74.65398
1,Roots Ocean Prime,restaurant,40.34333,-74.65944
2,Starbucks,cafe,40.34993,-74.65946


Ninety-two rows with four columns, `name`, `amenity`, `lat` and `lon`.

To pick out a single column you put its name in square brackets. Column names are matched exactly, including capital letters, and Python does not guess:

In [7]:
food["Amenity"]

KeyError: 'Amenity'

The last line reads `KeyError: 'Amenity'`.

**`KeyError` means you asked for something by name and that name is not there.** Here it is a capital A, where the actual column is called `amenity`. Notice how little help the message gives: it repeats what you asked for and does not suggest what you might have meant.

The habit that fixes this permanently is to look rather than to guess. Every table can tell you its own column names:

In [8]:
print(list(food.columns))
print(food["amenity"].value_counts().to_dict())

['name', 'amenity', 'lat', 'lon']
{'restaurant': 52, 'fast_food': 16, 'cafe': 15, 'ice_cream': 6, 'pub': 2, 'bar': 1}


Now we know both the column names and the values inside `amenity`, which is the vocabulary OpenStreetMap uses: 52 restaurants, 16 fast food places, 15 cafes, 6 ice cream shops, 2 pubs and 1 bar.

Note that `KeyError` also appears when you filter for a value that does not exist, except that it does not. Asking for `food[food["amenity"] == "pizza"]` returns an empty table and no error at all, because "no rows matched" is a perfectly valid answer. Silence is not always success, which is why we check counts rather than trusting that nothing complained.

## FileNotFoundError, i.e. Python cannot find that file

This one confuses beginners more than any other, because the file is usually sitting right there and can be seen in the file list.

The trap is that a path like `"data/food_table.csv"` is **relative**. It does not mean "somewhere on this computer", it means "starting from the folder this notebook lives in, go into `data`, then find `food_table.csv`".

So let us ask for something that is not there:

In [9]:
pd.read_csv("food_table.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'food_table.csv'

The last line is a `FileNotFoundError`, and the detail names exactly what it looked for.

The path we gave was `food_table.csv` with no folder in front, which means "in the same folder as this notebook". The file is actually one level down, inside `data`, which is why the correct path is `data/food_table.csv`.

**`FileNotFoundError` means the path is wrong, not that the file is missing.** Three things cause it, in rough order of frequency:

- The folder part is missing or wrong, as here.
- The notebook has been moved away from its `data` folder, or the `data` folder was not copied along with it. This is why the assignment asks you to submit the whole folder rather than the notebook alone.
- A spelling difference, including capital letters, since on some systems `Data` and `data` are different places.

If you are unsure where the notebook thinks it is, you can ask:

In [10]:
from pathlib import Path

print("this notebook is running in:", Path.cwd())
print("files next to it:", sorted(p.name for p in Path.cwd().iterdir() if not p.name.startswith(".")))

this notebook is running in: /Users/juergen/cloud/teaching/2026-CEE420-Urban-AI/code/F00
files next to it: ['00_01_notebooks_and_kernels.ipynb', '00_07_reading_errors.ipynb', 'data']


That prints the folder Python is working from and what is in it. If `data` is in that list, then `data/food_table.csv` will work.

## ModuleNotFoundError, i.e. the wrong Python

The last one looks like the others and means something quite different, so it is worth recognising.

In [11]:
import geopandaz

ModuleNotFoundError: No module named 'geopandaz'

`ModuleNotFoundError: No module named 'geopandaz'`.

Here it is a typo, with a z. But when this happens to you for real, the spelling will be correct, and the message will name a library you know perfectly well is part of this course, usually `geopandas`.

**When the spelling is right, `ModuleNotFoundError` almost never means the library is missing. It means you are running the wrong Python.** Your computer has its own Python, and the course environment inside the container has another, and only the second one has our libraries installed.

Two things to check, in this order:

- The bottom left corner of your editor should read `Dev Container: CEE420 Urban AI (FA26)`. If it does not, you are working outside the course environment.
- The kernel picker in the top right of the notebook should show the environment whose path is `/opt/venv/bin/python`. If it names something else, change it.

The fix is essentially never to install anything. If you find yourself about to run `pip install geopandas`, stop: that is the moment the two Pythons start to diverge and your setup stops matching everybody else's.

## The two problems that are not errors

Some failures produce no red text at all, which makes them harder, and both have appeared already in this notebook.

**Nothing happens when you run a cell.** No output, no number in the brackets, possibly a dialogue asking you to select a kernel. No kernel is attached, so nothing is running your code at all. Choose the environment at `/opt/venv/bin/python` and run the cell again.

**The answer is wrong but there is no complaint.** A filter that matches nothing returns an empty table. A number computed in the wrong units returns a number. An agent's code that measures from the wrong point returns a perfectly reasonable count. Python is not checking whether you asked a sensible question, and this is precisely why the precepts spend so much time on checking results rather than on avoiding errors.

The rule to take away: **a red error is a good day.** It is loud, it points at a line, and it names its own kind. The dangerous failures are the quiet ones.

## What to send when you are stuck

When you post in the course forum or ask a neighbour, three things make the difference between a five-minute answer and a long back and forth.

**One, the last line of the traceback.** Copy the text, not a photograph of your screen. The last line is the one that names the problem.

**Two, the cell you ran.** Copy the code, not a description of it.

**Three, what you expected.** One sentence. "I expected a count of restaurants" tells a reader far more than "it does not work", because most of the time the code did exactly what it was told and the disagreement is about what you meant.

Here is the whole thing as a template, and it is worth keeping:

```text
What I ran:
    food["Amenity"]

What I expected:
    the column of amenity types

What I got:
    KeyError: 'Amenity'
```

Note what is not on that list. You do not need to apologise, you do not need to explain that you are new to this, and you do not need a screenshot. Everybody in the room is producing errors today, including whoever answers you.

## Check your understanding

1. Your notebook says `NameError: name 'boundry' is not defined`. What are the two possible causes, and which is more likely if you have just restarted the kernel?
2. You get `FileNotFoundError` for a file you can see in the file list on the left. What is the most likely explanation?
3. A colleague says geopandas is not installed because they got a `ModuleNotFoundError`. What would you check before believing them?
4. Which is more dangerous, a cell that turns red or a cell that returns an empty table? Why?

## Where we are

You can read a traceback from the bottom, you recognise five kinds of problem on sight, and you know which two failures give no warning at all. That is genuinely most of what separates someone who gets stuck for an hour from someone who gets stuck for a minute.

Nothing in this notebook was about geography. All of it applies to every line of Python you write for the rest of the course.

## Further resources

Nothing in this course requires anything below.

The official Python tutorial has a chapter on errors and exceptions which lists many more kinds than the five here: https://docs.python.org/3/tutorial/errors.html. The pandas guide to indexing explains the selection rules behind `KeyError` in detail: https://pandas.pydata.org/docs/user_guide/indexing.html.

The table used in this notebook is an OpenStreetMap extract of Princeton, snapshot 2026-08-29, licensed ODbL 1.0, copyright OpenStreetMap contributors: https://www.openstreetmap.org/copyright.